In [113]:
# # Description
# 
# Prove that the Noise Reduction model works on a 2D polinomial.

# Add libs folder path to system variables to make the importation of local libraries easier.
# __________________________________________________________________________________________________
import os
import sys


CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)


# Import external and local libraries
# __________________________________________________________________________________________________
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import json
from typing import List

# Locals
from utils import *
from dataset import TensorDataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction


# Global / Path variables
# __________________________________________________________________________________________________
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')


# Make sure that the GPU is being used
# __________________________________________________________________________________________________
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [115]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets

X = X.drop(columns=['Sex'], inplace=False)

df_data = pd.DataFrame(X, columns=abalone.data.feature_names)
df_data['target'] = y

scaler = MinMaxScaler()
df_data = pd.DataFrame(scaler.fit_transform(df_data), columns=df_data.columns)

X = df_data.drop(columns=['target'], inplace=False)
y = df_data['target']

In [116]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pre-denoising XAI Benchmarck

In [117]:
model_params = {
    'ridge': {"alpha": 1.0},
    'pls': {"n_components": 1},
    'tree': {"max_depth": 5},
    'svm': {"dual": 'auto'},
    'knn': {"n_neighbors": 5, "weights": 'uniform', "algorithm": 'auto', "leaf_size": 30, "p": 2, "n_jobs": None},
}
xai_benchmark = XAI_benchmark(is_ts = False, model_params = model_params)

In [118]:
xai_benchmark.fit(X_train, y_train.values.ravel())

Fitting Ridge model...
Fitting Partial Least Squares model...
Fitting Decision Tree model...
Fitting Support Vector Machine model...
Fitting K-Nearest Neighbours model...
All models fitted!


/mnt/homeGPU/JJavierAR/repsol_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [119]:
xai_benchmark.predict(X_test, y_test.values.ravel(), get_metrics=True)

({'ridge': array([ 0.38003037,  0.3275612 ,  0.43924189,  0.39214949,  0.34038061,
          0.33834242,  0.31818762,  0.27100008,  0.22968997,  0.34825369,
          0.38118863,  0.23758119,  0.37401718,  0.24148964,  0.27414924,
          0.40956867,  0.40843408,  0.33975017,  0.47395012,  0.26962057,
          0.64823593,  0.26758322,  0.36580726,  0.37264421,  0.30940389,
          0.25590614,  0.32409632,  0.26229435,  0.32014026,  0.30898338,
          0.33824979,  0.37167707,  0.22961759,  0.31457676,  0.29328786,
          0.29407342,  0.20137304,  0.12366368,  0.25910555,  0.21811942,
          0.39917119,  0.5677922 ,  0.40025956,  0.30591948,  0.35541355,
          0.38227802,  0.30652937,  0.29525298,  0.38921719,  0.15909516,
          0.21778531,  0.60732457,  0.32468314,  0.14615722,  0.21936232,
          0.34905272,  0.35931662,  0.45979775,  0.3367447 ,  0.22910169,
          0.38059801,  0.44458516,  0.3436406 ,  0.24923187,  0.25361468,
          0.35727369,  0.1623

In [120]:
# Transform the data into a tensor
train_dataset = TensorDataset(
    x=X_train.values,
    y=y_train.values.reshape(-1, 1)
)
# Transform the data into a tensor
val_dataset = TensorDataset(
    x=X_test.values,
    y=y_test.values.reshape(-1, 1)
)

# Create the dataloaders
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [121]:
model = GridFullyDenseNN(
    n_layers=4,
    hidden_layers=[
        (X_train.shape[1], 1024),
        (1024, 512),
        (512, 1024),
        (1024, 1),
    ],
    dropout_layers=[
        0.0,
        0.0,
        0.0,
        0.0,
    ],
    activation_func_layers=[
        nn.Sigmoid(),
        nn.Sigmoid(),
        nn.Sigmoid(),
        nn.Sigmoid(),
    ],
    want_dropout=[
        False,
        False,
        False,
        False,
    ],
    want_linear=[
        True,
        True,
        True,
        True,
    ],
    want_activation=[
        True,
        True,
        True,
        True,
    ],
)
model.to(device)

GridFullyDenseNN(
  (model): Sequential(
    (linear_0): Linear(in_features=7, out_features=1024, bias=True)
    (activation_0): Sigmoid()
    (linear_1): Linear(in_features=1024, out_features=512, bias=True)
    (activation_1): Sigmoid()
    (linear_2): Linear(in_features=512, out_features=1024, bias=True)
    (activation_2): Sigmoid()
    (linear_3): Linear(in_features=1024, out_features=1, bias=True)
    (activation_3): Sigmoid()
  )
)

In [122]:
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Define the trainer
trainer_basic = Trainer(
    model=model,
    train_generator=train_dataloader,
    val_generator=val_dataloader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=500,
    checkpoints_path=os.path.join(CHECKPOINT_PATH, 'abalone_gradient')
)

Train the model

In [123]:
type(y), y.shape

(pandas.core.series.Series, (4177,))

In [124]:
model, train_losses, val_losses, best_train_loss, best_val_loss = trainer_basic.fit(verbose=True)

Training is started!


Epochs loop:  28%|▎| 140/500 [01:49<04:40,  1.28epoch/s, Train loss: 0.006128079980300

Training stopped as validation loss did not improve for 15 epochs.


or load a checkpoint

In [125]:
# model.load_state_dict(torch.load(os.path.join(CHECKPOINT_PATH, abalone_gradient.pth')))

In [126]:
y_pred_test = model(torch.tensor(X_test.values).float().to(device)).cpu().detach().numpy().reshape(-1)

In [127]:
# Show the metrics
gt_values = y_test.values
predicted_values = y_pred_test

mae = mean_absolute_error(gt_values, predicted_values)
mse = mean_squared_error(gt_values, predicted_values)
rmse = np.sqrt(mse)
# mape = np.mean(np.abs(gt_values - predicted_values) / gt_values) * 100
r_squared = r2_score(gt_values, predicted_values)
nn_benchmark = {
    'nn': {
        "MSE: ": mse,
        "RMSE: ": rmse,
        "MAE: ": mae,
        # "MAPE: ": mape,
        "R2: ": r_squared
    }
}
nn_benchmark

{'nn': {'MSE: ': 0.006144430739110155,
  'RMSE: ': 0.07838641935380232,
  'MAE: ': 0.0563137346085464,
  'R2: ': 0.5549985931737877}}

# Gradients to reduce noise

In [128]:
X_no_noise = X.copy()
y_no_noise = y.copy()

In [129]:
dlnr = DLNoiseReduction(model=model, criterion=criterion)
dlnr.fit(X.values, y.values.reshape(-1, 1))

In [130]:
X_no_noise, y_no_noise = dlnr.transform(
                                        nrr=0.05,
                                        nr_threshold=0.01,
                                        max_epochs=200,
                                        plot_progress=False,
                                        path_to_save_imgs=None,
                                        denoise_y=True
)

# Denoised XAI Benchmark

In [131]:
X_no_noise.shape, y_no_noise.shape

((4177, 7), (4177, 1))

In [132]:
xai_benchmark.fit(X_no_noise, y_no_noise.ravel())

Fitting Ridge model...
Fitting Partial Least Squares model...
Fitting Decision Tree model...
Fitting Support Vector Machine model...
Fitting K-Nearest Neighbours model...
All models fitted!


/mnt/homeGPU/JJavierAR/repsol_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [133]:
xai_benchmark.predict(X_no_noise, y_no_noise.ravel(), get_metrics=True)

({'ridge': array([0.37960705, 0.22247173, 0.31635134, ..., 0.32794439, 0.32453927,
         0.36182056]),
  'pls': array([0.28480987, 0.22681008, 0.30489704, ..., 0.37843301, 0.36490557,
         0.45416309]),
  'decision_tree': array([0.43009963, 0.20773874, 0.33412998, ..., 0.36407   , 0.349452  ,
         0.41680137]),
  'svm': array([0.38805776, 0.21805265, 0.31133703, ..., 0.33390829, 0.31463177,
         0.36204311]),
  'knn': array([0.42517302, 0.20860394, 0.3100706 , ..., 0.33056333, 0.31596704,
         0.3700571 ])},
 {'ridge': {'mse': 0.0005484334165286177,
   'rmse': 0.023418655310000563,
   'mae': 0.01719941981487855,
   'R2': 0.9428445979714416},
  'pls': {'mse': 0.004657346493303529,
   'rmse': 0.0682447543281059,
   'mae': 0.050979826255005634,
   'R2': 0.5146311235081902},
  'decision_tree': {'mse': 0.0011983175242279607,
   'rmse': 0.0346167231873261,
   'mae': 0.02513324598521392,
   'R2': 0.875116435667552},
  'svm': {'mse': 0.0005328779828815118,
   'rmse': 0.02308

In [134]:
xai_benchmark.predict(X_test.values, y_test.values, get_metrics=True)

({'ridge': array([ 0.38860513,  0.33233458,  0.45767781,  0.39836079,  0.34740629,
          0.34873211,  0.32397889,  0.26605643,  0.22783478,  0.3496765 ,
          0.39133077,  0.23895137,  0.37820436,  0.23970217,  0.27321317,
          0.42111091,  0.41703385,  0.34175126,  0.48613912,  0.26853058,
          0.67071636,  0.26188058,  0.3692628 ,  0.3782418 ,  0.309156  ,
          0.2559111 ,  0.32920765,  0.25941497,  0.32497473,  0.31384459,
          0.33926561,  0.37662682,  0.22794181,  0.32212461,  0.29725537,
          0.2891841 ,  0.19767999,  0.11120148,  0.25746004,  0.2133967 ,
          0.39826796,  0.58971794,  0.40381056,  0.30842349,  0.35625037,
          0.37423185,  0.30464023,  0.29278248,  0.39274869,  0.15833123,
          0.21422859,  0.62526953,  0.32641266,  0.13829979,  0.21683712,
          0.35307436,  0.36517308,  0.46785952,  0.34171879,  0.228597  ,
          0.38292756,  0.45818494,  0.34650257,  0.24739767,  0.24699898,
          0.35798096,  0.1548